<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
《<a href="https://mng.bz/lZ5B">从零构建推理模型</a>》一书的配套代码，作者：<a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库： <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 第 7 章：改进强化学习中的 GRPO

本 notebook 使用的软件包：

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # 供 reasoning_from_scratch 使用
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F01_raschka.webp?1" width=600>

&nbsp;
## 7.1 改进 GRPO

- 注意：如果本章技术细节过多，而你对 GRPO 的进一步内容不感兴趣，可以跳过本章；下一章不依赖本章内容。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F02_raschka.webp?1" width=400>

&nbsp;
## 7.2 跟踪 GRPO 性能指标

- 上一章进行了一次简短的 RLVR 训练，并简要讨论了结果。

- 例如，上一节看到的短期运行输出采用以下结构：

```
[Step 1/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=5.5
[Step 2/50] loss=-0.0000 reward_avg=0.000 avg_resp_len=6.8
[Step 3/50] loss=0.3592 reward_avg=0.250 avg_resp_len=7.8
[Step 4/50] loss=2.7401 reward_avg=0.250 avg_resp_len=56.5
[Step 5/50] loss=3.3214 reward_avg=0.500 avg_resp_len=251.2
# ...
```

- 本节将接着之前的内容，讨论运行 RLVR 时应跟踪的一些指标


&nbsp;
### 7.2.1 执行一次 GRPO 训练

- 第 6 章的代码试图在展示完整 GRPO 流程和保持简洁、提升可读性之间取得平衡
- 附加材料包含一个代码相同的脚本（[ch06/02_rlvr_grpo_scripts_intro/rlvr_grpo_original_no_kl.py](https://github.com/rasbt/reasoning-from-scratch/blob/main/ch06/02_rlvr_grpo_scripts_intro/rlvr_grpo_original_no_kl.py)），从终端运行可能更方便
- 本章将引用附加材料中的脚本，以便使用并避免大量重复代码使 notebook 和章节显得杂乱
- 本章会使用以下辅助函数，从附加材料下载代码并保存到本地目录


In [ ]:
from pathlib import Path
import requests

def download_from_github(rel_path, out=None):
    github_raw_base = (  # 基础 URL
        "https://raw.githubusercontent.com/rasbt/"
        "reasoning-from-scratch/refs/heads/main/"
    )

    rel_path = Path(rel_path)
    # 使用 URL 中的文件名作为默认输出文件名
    out = Path(out) if out is not None else Path(rel_path.name)

    # 如果本地文件已存在，则跳过下载
    if out.exists():
        size_kb = out.stat().st_size / 1e3
        print(f"{out}: {size_kb:.1f} KB (cached)")
        return out

    # 下载文件
    r = requests.get(github_raw_base + rel_path.as_posix())
    r.raise_for_status()

    out.write_bytes(r.content)
    size_kb = out.stat().st_size / 1e3
    print(f"{out}: {size_kb:.1f} KB")

- 下载 GRPO 脚本（基于第 6 章代码）：

In [ ]:
download_from_github(
    "ch06/02_rlvr_grpo_scripts_intro/rlvr_grpo_original_no_kl.py"
)

- 如果下载文件的大小远小于上面显示的数值，下载过程可能出了问题
- 请仔细检查 URL 是否输入正确
- 如果文件下载仍有问题，请参阅[故障排除指南](https://github.com/rasbt/reasoning-from-scratch/blob/main/troubleshooting.md)


- 训练代码的计算成本较高，如果不想亲自运行也很合理；附加材料提供了以下运行的日志文件（稍后会介绍如何下载）：

```bash
uv run rlvr_grpo_original_no_kl.py \
    --steps 500  \
    --max_new_tokens 1024
```


- **提示：** 可以在上面的执行命令中加入 `--show_eta`，显示针对当前计算机估算的脚本总运行时间。

- 建议在终端中运行该脚本（使用上面的命令）；也可以在 Jupyter notebook 的代码单元中运行，只需在前面加上“!”，即 `!uv run rlvr_grpo_original_no_kl.py`
- 如果不使用 `uv`，请将 `uv run` 替换为 `python`，即 `python rlvr_grpo_original_no_kl.py`


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F03_raschka.webp" width=600>

- 如前所述，亲自复现上述运行的成本可能很高，可以改为按如下方式下载日志文件：

In [ ]:
# 纯文本格式的日志文件

download_from_github(
    "ch07/02_logs/ch06_rlvr_grpo_original_no_kl_metrics.txt"
)

# 同一日志文件的 CSV 格式版本
download_from_github(
    "ch07/02_logs/ch06_rlvr_grpo_original_no_kl_metrics.csv"
)

&nbsp;
### 7.2.2 检查 GRPO 训练过程

- 本章将使用以下辅助函数绘制日志文件中的结果。

In [ ]:
import csv
import matplotlib.pyplot as plt


def moving_average(values, window_fraction=0.25):
    # 平滑含噪训练信号，以呈现训练中的长期趋势
    window_size = max(1, int(window_fraction * len(values)))
    smoothed = []

    for i in range(len(values)):
        start_idx = max(0, i - window_size + 1)
        window_mean = sum(values[start_idx : i + 1]) / (i - start_idx + 1)
        smoothed.append(window_mean)

    return smoothed


def plot_grpo_metrics(csv_path, columns, save_as=None):
    data = {name: {"steps": [], "values": []} for name in columns}

    # 打开并读取 CSV 日志文件
    with Path(csv_path).open(newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if not row or not row.get("step"):
                continue

            # 将训练步数作为所有指标共用的 x 轴
            step = int(row["step"])

            for name in columns:
                value_str = row.get(name)
                if value_str:
                    data[name]["steps"].append(step)
                    data[name]["values"].append(float(value_str))

    # 创建固定网格，以便并排展示损失、奖励、回复长度等指标
    fig, axes = plt.subplots(2, 2, sharex=True, figsize=(6, 4))
    axes = axes.ravel()

    for i, name in enumerate(columns):
        steps = data[name]["steps"]
        values = data[name]["values"]

        # 跳过不存在的指标
        if not values:
            fig.delaxes(axes[i])
            continue

        # 评估准确率使用条形图，因为并非每一步都有数据
        if name == "eval_acc":
            axes[i].bar(steps, values, width=20)
        else:
            axes[i].plot(steps, values, alpha=0.4)
            axes[i].plot(steps, moving_average(values))

        axes[i].set_ylabel(name)

    for j in (2, 3):
        if axes[j] in fig.axes:
            axes[j].set_xlabel("Step")

    plt.tight_layout()
    if save_as is not None:
        plt.savefig(save_as)
    plt.show()


# 绘制 GRPO 训练过程
plot_grpo_metrics(
    "ch06_rlvr_grpo_original_no_kl_metrics.csv",
    columns=["loss", "reward_avg", "avg_response_len", "eval_acc"],
    # save_as="4.pdf"
)

- 以下是一些总体观察和思考：
  - 平均回复长度最初应上升（理想情况下准确率也应上升）；总体看来表现不错，但在第 400 步前后出现了一些下降
  - 与预训练相比，这里的损失意义不大；它应相对平稳，允许一定波动（但接近末尾的尖峰值得警惕）
- 平均奖励应上升；从技术上说，平均奖励达到 1.00 表示所有答案都正确，这当然很好，但此时已没有训练信号，应停止训练
- 目标或外部任务性能（这里是 MATH-500 准确率）应提高；它最初确实提高，随后下降


- 总结：早期提升很快，随后收益递减
  - 原因之一是算法不够稳定
  - （另一种可能解释是问题变难了，但这样就不应看到评估准确率下降）


- 这里侧重对 GRPO 运行进行简化跟踪；下一节会增加更多指标
- 顺便说明，默认不会跟踪评估准确率（这里指 MATH-500 准确率）
- 可以加入 `--eval_on_checkpoint 500`，在训练期间对 MATH-500 的 500 个测试样本计算准确率，但这会减慢训练
- 也可以在训练后使用 `evaluate_math500.py` 脚本（包含第 3 章代码）计算


In [ ]:
download_from_github(
    "ch03/02_math500-verifier-scripts/evaluate_math500.py"
)

- 随后可以在给定检查点上按如下方式运行评估：

```bash
uv run evaluate_math500.py \
  --dataset_size 500 \
  --checkpoint_path \
    "checkpoints/rlvr_grpo_original_no_kl/\
qwen3-0.6B-rlvr-grpo-step00050.pth"
```

- 我已经把这些评估运行每隔 50 步的结果加入了之前分析的日志文件。

&nbsp;
## 7.3 跟踪更高级的 GRPO 性能指标

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F05_raschka.webp" width=400>

- 上一节查看了用于跟踪 GRPO 运行的基础性能指标
- 还可以跟踪其他有用指标
- 两个示例包括：
  - 第 6 章 `compute_grpo_loss` 中已经计算的优势值
  - 所计算序列的熵


&nbsp;
### 7.3.1 跟踪优势值

- 作为 GRPO 算法的一部分，我们会计算第 6 章讨论过的优势值
- 这些优势值也可用于分析训练动态


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F06_raschka.webp" width=400>

- 具体来说，可以从优势值派生两个新的汇总统计量：平均值（样本均值）和标准差（std）。

In [ ]:
import torch

def compute_advantage_stats(rewards_list):
    # 这是 GRPO 中已经计算的内容：
    rewards = torch.tensor(rewards_list)
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

    # 下面是新增的统计量：
    adv_avg = advantages.mean().item()
    adv_std = advantages.std().item()

    return advantages, adv_avg, adv_std

- 下面使用四个奖励值的小样本进行示例（类似上一幅图中的数值）。

In [ ]:
adv, adv_avg, adv_std = compute_advantage_stats([1., 1., 0., 0.])

print(f"Advantages: {adv}")
print(f"Advantage mean = {adv_avg:.4f}, std = {adv_std:.4f}")

- 平均值：
  - 按照这里计算优势值的方式，其平均值始终为 0；在 GRPO 中，这只是一项健全性检查，用于确认代码按预期工作

- 标准差：
  - 这里的标准差包含更多信息
  - `std ≈ 1` 表示梯度信号的尺度良好，适合稳定更新
  - `std ≪ 1` 表示信号正在消失（奖励塌缩时经常发生）
  - `std ≫ 1` 表示更新过于尖锐，可能使训练不稳定


- 如果一组中的所有奖励都非常相似，优势值的标准差会接近 0；这有助于发现模型缺少有效训练信号的情况。

In [ ]:
adv, adv_avg, adv_std = compute_advantage_stats([0., 0., 0., 0.])
print(f"Advantages: {adv}")
print(f"Advantage mean = {adv_avg:.4f}, std = {adv_std:.4f}")

In [ ]:
adv, adv_avg, adv_std = compute_advantage_stats([1., 1., 1., 1.])
print(f"Advantages: {adv}")
print(f"Advantage mean = {adv_avg:.4f}, std = {adv_std:.4f}")

- 应结合平均奖励查看优势值统计量
- 例如，平均奖励为 1 是好事，但也意味着组内没有剩余的相对训练信号；此时可以停止训练，或改用更难的样本


&nbsp;
### 7.3.2 跟踪熵

- 熵用于衡量模型在预测下一个词元时的不确定性
- 高熵表示概率质量分散在多个词元上，通常对应更多探索
- 低熵表示概率质量集中在一个词元上，生成更确定；过低则可能意味着策略塌缩


- 在计算熵之前，先回顾一下第 5 章中的对数概率计算。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F07_raschka.webp" width=600>

- 为便于清晰绘图，下面使用词表大小为 7 的玩具 logits，而不是约含 151,000 个词元的真实词表。

In [ ]:
# 示例 logits
logits = torch.tensor([
    0.6667, -2.0000,  1.3333, -0.0000, -0.6667,  2.0000, -1.3333
])

- 以下代码实现了上一幅图中的计算。

In [ ]:
logprobs = torch.log_softmax(logits, dim=-1)
print("All token logprobs:", logprobs)

# 所选词元的索引
selected_token = torch.argmax(logprobs)

selected = logprobs[selected_token]
print("Selected token ID:", selected_token)
print("Selected token logprob:", selected)

- 熵通过将各词元的概率乘以其对数概率并求和得到
- 也可以直接跟踪这些项的总和，不过熵是一个成熟且更容易解释的不确定性指标


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F08_raschka.webp" width=400>

- 一个实用的经验判断是：
  - **极低熵（约 0–0.5）：** 一个词元占据主导，模型非常自信或接近确定性
  - **中等熵（约 1–2）：** 概率由少数词元共享，通常适合稳定训练
  - **高熵（远大于 2，接近 `log(vocab_size)`）：** 概率分布很分散，模型更随机或不确定；本例中 `log(7) = 1.9459`


- 可以按如下方式计算熵。

In [ ]:
# 根据 logits 计算概率
probs = torch.softmax(logits, dim=-1)
logprobs = torch.log_softmax(logits, dim=-1)

# 根据概率计算熵
entropy = torch.sum(-(probs * logprobs))

print("Probs:", probs)
print("Entropy:", entropy)

- 或者，如果只有对数概率值，可以通过 `torch.exp` 将其转换回概率值
- 可以看到，下面的数值与上面的 `probs` 数值一致


In [ ]:
print("Probs:", torch.exp(logprobs))

- 接下来将 `sequence_logprob` 升级为 `sequence_logprob_and_entropy`，使其同时返回序列对数概率和熵。

In [ ]:
def sequence_logprob_and_entropy(model, token_ids, prompt_len):
    # 原有代码：与第 5 章相同
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)

    targets = token_ids[1:]
    selected = logprobs[:-1].gather(1, targets.unsqueeze(-1)).squeeze(-1)

    # 生成答案词元的对数概率（对答案步骤求和）
    selected_answer_logprobs = selected[prompt_len - 1:]
    logp_all_steps = torch.sum(selected_answer_logprobs)

    # 新增：计算熵
    all_answer_logprobs = logprobs[:-1][prompt_len - 1:]
    if all_answer_logprobs.numel() == 0:  # 防止模型立即返回 EOS 词元
        entropy_all_steps = logp_all_steps.new_tensor(0.0)
    else:
        all_answer_probs = torch.exp(all_answer_logprobs)  # 将对数概率转换为概率
        plogp = all_answer_probs * all_answer_logprobs     # 逐元素计算 p * log p
        step_entropy = -torch.sum(plogp, dim=-1)           # 对词表维度求和，得到每一步的熵
        entropy_all_steps = torch.mean(step_entropy)       # 对答案步骤取平均值

    return logp_all_steps, entropy_all_steps

- 然后可以在 `compute_grpo_loss` 中使用该函数
- 图中展示的是单个词元的 `step_entropy`，而该函数会对答案中的所有词元取平均
- 例如，对于 `"this is the LLM response"`，它会返回这些回复词元的平均熵


&nbsp;
### 7.3.3 绘制额外的 GRPO 指标

- 下面实际运行一次，并分析训练过程中的优势值统计量和熵
- 为此，可以在 `compute_grpo_loss` 中替换 `sequence_logprob_and_entropy`，再在内部使用 `compute_grpo_loss` 的 `train_rlvr_grpo` 中输出并跟踪熵
- 为避免大段重复代码，完整修改版位于附加材料中，可以按如下方式下载


In [ ]:
download_from_github(
    "ch07/03_rlvr_grpo_scripts_advanced/7_3_plus_tracking.py"
)

- 可以像前面的脚本一样运行它。

```bash
uv run 7_3_plus_tracking.py \
    --steps 500  \
    --max_new_tokens 1024
```

- 由于训练耗时较长，也可以直接下载预先生成的 CSV 日志文件。

In [ ]:
download_from_github(
    "ch07/02_logs/7_3_plus_tracking_metrics.csv"
)

In [ ]:
plot_grpo_metrics(
    "7_3_plus_tracking_metrics.csv",
    columns=["reward_avg", "adv_avg", "adv_std", "entropy_avg"],
    # save_as="9.pdf"
)

- 平均优势值始终为零，这正是 GRPO 风格归一化的预期结果
  - 优势值相对于组均值计算，因此按构造其总和应为零
  - 该指标在 GRPO 中主要用于健全性检查；如果它偏离零，可能表示代码存在缺陷或归一化存在偏差
- 优势值标准差是信息量更大的信号
  - 训练早期方差相对较高，表示模型会生成多样化的 rollout
  - 随后优势值标准差缓慢下降并趋于稳定，表示 rollout 的质量逐渐接近
  - 归根结底，非零且稳定的优势值标准差表示仍然存在可用的学习信号


- 接下来查看熵
  - 训练早期熵相对平稳且较低（这表示 LLM 的输出相当确定）
  - 实际使用时，提高采样温度会增加 rollout 期间的输出多样性，但不会改变 LLM 底层分布的熵
  - 训练后期（约第 200 步以后）熵显著上升，表示下一词元分布变宽，模型变得更具随机性
  - 熵非常低可能是塌缩的迹象，即模型反复生成相同或非常相似的输出
  - 不过在此次运行中，将熵与平均奖励以及不消失的优势值标准差结合考虑，表明整体上是健康的探索，而不是塌缩


- 每项指标反映的信息不同，因此应结合训练上下文共同解读。

&nbsp;
## 7.4 使用裁剪后的策略比率稳定序列级 GRPO

- 到目前为止，我们主要关注分析 GRPO 的训练结果
- 接下来开始扩展 GRPO 算法
- 当前使用的是 GRPO 的简化版本；本节重点介绍如何在 GRPO 损失中使用裁剪后的策略比率


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F10_raschka.webp" width=400>

&nbsp;
### 7.4.1 计算裁剪后的策略比率

- 裁剪后的策略比率用于衡量当前策略（正在训练的 LLM）与同一策略较早版本之间的差异
- 具体来说，它比较更新步骤之前和之后计算的序列对数概率
- 在下图的 GRPO 流程中，这对应于比较第 4 步使用旧参数计算的对数概率，与第 6 步更新模型后得到的对数概率


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F11_raschka.webp?1" width=400>

- 回顾一下，上一章按如下方式计算策略梯度损失：

In [ ]:
# 计算 rollout
# ...

# 计算奖励
rewards = torch.tensor([1., 1., 0., 0.])

# 计算序列对数概率
logprobs = torch.tensor([-7.9243, -20.1546, -16.6130, -23.3677])

advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

pg_loss = -(advantages.detach() * logprobs).mean()
print("Policy gradient loss:", pg_loss)

- 裁剪比率由旧策略的对数概率（`old_logps`）和当前对数概率（`new_logps`）计算得到
- 推导过程超出本书范围，感兴趣的读者可参阅论文 [Proximal Policy Optimization Algorithms](https://arxiv.org/pdf/1707.06347)
- 为便于说明，下面将上例的对数概率用作 `old_logps`，假设已经执行了一次模型更新，再用更新后的模型计算 `new_logps`（为进行公平比较，旧策略和当前策略应使用相同提示词）


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F12_raschka.webp?2" width=600>

In [ ]:
new_logps = logprobs

# new_logps 的值
# 在注释中并排显示
old_logps = torch.tensor([
    -10.9243,   # -7.9243
    -20.3546,   # -20.1546
    -14.6130,   # -16.6130
    -23.3677,   # -23.3677
])

log_ratio = new_logps - old_logps
ratio = torch.exp(log_ratio)
clip_eps = 10.0
clipped_ratio = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps)

print("Ratio:        ", ratio)
print("Clipped ratio:", clipped_ratio)

- `ratio` 表示新旧对数概率的差异程度；如果二者相同，结果就是 1.0
- `clipped_ratio` 限制相对于旧策略的最大偏移，使更新保持相对保守，从而提高训练稳定性
- 通常会将其限制在 1 ± `clip_eps`；DeepSeek-R1 训练中将 `clip_eps` 设为 10，意味着裁剪很弱（而在其他强化学习训练流程中，也常使用 0.1 这样小得多的值）


- 接下来将 `ratio` 和 `clipped_ratio` 应用于损失计算
- 之前是直接将优势值与对数概率相乘
- 现在使用策略比率缩放优势值，并通过裁剪目标限制每个 rollout 对更新的影响程度


In [ ]:
# 将优势值视为固定学习信号（不通过奖励反向传播）
adv = advantages.detach()

unclipped = ratio * adv
clipped = clipped_ratio * adv

# PPO 风格的裁剪
obj = torch.minimum(unclipped, clipped)

    
clipped_pg_loss = -torch.mean(obj)
policy_ratio = torch.mean(ratio)

print("Clipped policy gradient loss:", clipped_pg_loss)
print("Policy ratio:", policy_ratio)

- 本例中的策略比率很大，意味着更新后的策略（模型）为采样词元分配了比旧策略高得多的概率
- 请注意，裁剪后的策略梯度损失（-2.3998）略低于之前计算的常规策略梯度损失（-2.5764），这意味着模型权重更新幅度也略小
- 这种裁剪可以防止过于激进的更新破坏训练稳定性


- 作为参考，不同方法在 GRPO 中分别使用词元级或序列级概率，常见变体如下：

| 论文 | 类型 | 表示法 |
|---|---|---|
| [DeepSeekMath GRPO](https://arxiv.org/abs/2402.03300)，2024 | 词元级 | `r_{i,t}(theta) = pi_theta(o_{i,t} \| q, o_{i,<t}) / pi_theta_old(o_{i,t} \| q, o_{i,<t})` |
| [DeepSeek-R1 GRPO](https://arxiv.org/abs/2501.12948)，2025 | 序列级 | `r_i(theta) = pi_theta(o_i \| q) / pi_theta_old(o_i \| q)` |
| [DAPO](https://arxiv.org/abs/2503.14476)，2025 | 词元级 | `r_{i,t}(theta) = pi_theta(o_{i,t} \| q, o_{i,<t}) / pi_theta_old(o_{i,t} \| q, o_{i,<t})` |
| [GSPO](https://arxiv.org/abs/2507.18071)，2025 | 序列级 | `s_i(theta) = (pi_theta(y_i \| x) / pi_theta_old(y_i \| x))^(1 / \|y_i\|)` |


&nbsp;
### 7.4.2 使用裁剪后的策略比率训练

- 可以从附加材料下载更新后的 GRPO 训练脚本，其中已实现裁剪后的策略比率计算：

In [ ]:
download_from_github(
    "ch07/03_rlvr_grpo_scripts_advanced/7_4_plus_clip_ratio.py"
)

- 与之前类似，可以按如下方式运行：

```bash
uv run 7_4_plus_clip_ratio.py \
    --steps 500  \
    --max_new_tokens 1024
```


- DeepSeek-R1 会生成一个很大的 rollout 池（8,192 个），将其拆分成 16 个小批次，然后在这些小批次上执行一个内部训练轮次
- 具体流程如下：
  1. 复制当前模型作为参考策略
  2. 使用参考策略采样一个大型 rollout 池
  3. 将固定的 rollout 拆分成小批次
  4. 对每个小批次：
      - 在参考模型下计算 `old_logps`
      - 在当前模型下计算 `new_logps`（当前模型会在每次小批次更新后发生变化）
      - 更新当前模型
  5. 更新参考模型


- 由于资源限制，我们无法生成如此多的 rollout 和小批次，因此需要采用折中方案：
1. 复制当前模型作为参考策略
2. 使用参考策略采样 8 个 rollout
3. 然后计算：
  - 参考模型下的 `old_logps`
  - 当前模型下的 `new_logps`
  - 更新当前模型
4. 使用不同的 rollout 重复第 2、3 步，而不是使用小批次
5. 更新参考模型


- 简而言之，DeepSeek-R1 一次生成巨大的 rollout 池，再按小批次依次处理
- 我们的代码则多次生成较小批量的 rollout


- 可以按如下方式下载并可视化此次运行的日志文件：

In [ ]:
download_from_github(
    "ch07/02_logs/7_4_plus_clip_ratio_metrics.csv",
)

In [ ]:
plot_grpo_metrics(
    "7_4_plus_clip_ratio_metrics.csv",
    columns=["loss", "reward_avg", "avg_response_len", "eval_acc"],
    save_as="13_1.pdf"
)

In [ ]:
plot_grpo_metrics(
    "7_4_plus_clip_ratio_metrics.csv",
    columns=["policy_ratio", "adv_avg", "adv_std", "entropy_avg"],
    save_as="13_2.pdf"
)

- 从上图可以看出，训练现在比以前更稳定，第 400 步附近不再出现性能下降。

&nbsp;
## 7.5 使用 KL 项控制模型变化幅度

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F14_raschka.webp?2" width=400>

- 上一章篇幅已经很长，因此跳过了 GRPO 算法中的 KL 损失项
- 熟悉强化学习算法的读者可能已经看出，我们实质上实现了带有组归一化优势值的 [REINFORCE](https://en.wikipedia.org/wiki/Policy_gradient_method#REINFORCE)
- 为补全 GRPO 算法，现在加入 KL 损失项
- KL 是 Kullback-Leibler divergence（KL 散度）的缩写，用于衡量当前策略（正在训练的 LLM）偏离参考策略（训练开始时的原始 LLM）的程度
- 它作为正则项抑制过大的策略更新，使模型保持接近原始行为，并防止剧烈且不稳定的更新
- 这与前面实现的裁剪策略比率并非毫无关系：之前只关注单次更新的幅度，现在则显式抑制长期漂移


&nbsp;
### 7.5.1 实现 KL 损失项

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F15_raschka.webp?2" width=400>

- 对每个 rollout，使用完全相同的生成词元，分别在当前策略（模型）和固定参考模型下计算对数概率，如图右侧所示
- 随后对这些对应对数概率之差求和来计算 KL 散度，从而量化当前策略相对于参考策略的偏移程度
- 该 KL 项会加入策略梯度损失，因此如果某次权重更新虽然提高了奖励，却也大幅增加了与参考模型之间的差异（散度），训练时就会受到惩罚
- 这里的参考模型是训练开始前的原始模型（在 DeepSeek-R1 中，每 400 步会用最新模型更新一次参考模型）


- 为避免代码重复，下面不会展示完整的 `compute_grpo_loss_with_kl`，而只关注需要对 `compute_grpo_loss` 函数做出的修改
- 新增部分由 `"# New"` 标出，或位于 `if kl_coeff` 条件块中：


In [ ]:
import copy

# kl_coeff = 0.0 会禁用 KL 项，此时代码与之前类似
kl_coeff = 0.0  

# 复制原始模型并禁用梯度，防止其被更新
if kl_coeff:
    ref_model = copy.deepcopy(model).to(device)  # noqa: F821
    ref_model.eval()
    for p in ref_model.parameters():
        p.requires_grad = False
else:
    ref_model = None


def compute_grpo_loss_with_kl(
    model,
    ref_model,  # 新增
    # ...
    kl_coeff=0.02,  # 新增
):
    roll_logps, roll_ref_logps, roll_rewards, samples = [], [], [], []  # noqa: F841
    # ...

    for _ in range(num_rollouts):  # noqa: F821
        token_ids, prompt_len, text = sample_response(  # noqa: F821
        # ...
        )

        # 使用参考模型计算对数概率
        if kl_coeff:
            with torch.no_grad():
                ref_logp = sequence_logprob(  # noqa: F821
                    ref_model, token_ids, prompt_len
                )
        else:
            ref_logp = None

        reward = reward_rlvr(text, example["answer"])  # noqa: F821

        roll_rewards.append(reward)
        roll_logps.append(logp)  # noqa: F821
        if kl_coeff:
            roll_ref_logps.append(ref_logp)
        
        # ...

    rewards = torch.tensor(roll_rewards, device=device)  # noqa: F841
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

    logps = torch.stack(roll_logps)
    if kl_coeff:
        ref_logps = torch.stack(roll_ref_logps).detach()

    pg_loss = -(advantages.detach() * logps).mean()

    # 计算 KL 损失项
    if kl_coeff:
        kl_loss = kl_coeff * torch.mean(logps - ref_logps)
    else:
        kl_loss = torch.tensor(0.0, device=logps.device)

    # 将 KL 损失项加入策略梯度损失
    loss = pg_loss + kl_loss  # noqa: F841

- 请注意，加入 KL 损失项会增加资源需求，因为现在还需要保留一份原始模型副本
- `kl_coeff` 越大，散度惩罚越强


&nbsp;
### 7.5.2 使用 KL 损失项训练

- 可以按如下方式从附加材料下载完整脚本：

In [ ]:
download_from_github(
    "ch07/03_rlvr_grpo_scripts_advanced/7_5_plus_kl.py"
)

- 与之前类似，可以按如下方式运行：

```bash
uv run 7_5_plus_kl.py \
    --steps 500  \
    --max_new_tokens 1024
```


- 与之前类似，运行实验可能耗费较多资源和成本，因此可以下载以下日志文件进行分析：

In [ ]:
download_from_github(
    "ch07/02_logs/"
    "7_5_plus_kl_metrics.csv"
)

In [ ]:
plot_grpo_metrics(
    "7_5_plus_kl_metrics.csv",
    columns=["loss", "reward_avg", "avg_response_len", "eval_acc"],
    # save_as="16.pdf"
)

In [ ]:
plot_grpo_metrics(
    "7_5_plus_kl_metrics.csv",
    columns=["policy_ratio", "adv_avg", "adv_std", "entropy_avg"],
)

- 可以看到，训练在第 50 步后不久就开始崩溃，评估准确率逐渐降至 0
- 手动检查生成的模型回复可以发现，模型开始输出无意义文本和类似 `"framework.raises Self profess(import"` 的随机词元
- 这可能有多个原因：
1. KL 项基于求和后的对数概率计算
   - `kl_loss = kl_coeff * mean(new_logps - ref_logps)`，其中 `new_logps` 和 `ref_logps` 都是对答案词元求和的结果
   - 因此，序列越长，KL 项的绝对值就越大；这可能会激励模型生成很长的输出（从不断增长的回复长度也能看出），并使训练不稳定
2. 奖励在第 200 步附近崩溃后，KL 成为唯一的梯度来源
   - 当所有奖励均为零时，优势值也变为 0，因此策略梯度损失不再产生梯度
   - 但 KL 项仍然存在，继而主导更新，并可能将模型推向更高熵或接近均匀的分布；这解释了生成答案中的随机性（以及 0.0% 的评估准确率）


- KL 项是 GRPO 的标准组成部分，因此本节仍予以介绍；不过，多项研究发现不使用它时模型可能训练得更好，例如：
  - [Dr. GRPO](https://arxiv.org/abs/2503.20783)
  - [Olmo 3](https://arxiv.org/abs/2512.13961)
  - [DeepSeek V3.2](https://arxiv.org/abs/2512.02556)


也可以通过其他方式稳定训练：

1. 使用词元级对数概率，对对数概率和 KL 按长度归一化
   - 要使用词元级对数概率，可在计算策略比率和 KL 项之前除以生成长度
   - 这样可以消除鼓励模型扩大序列长度的因素

2. 降低 `kl_coeff`（例如降至 0.001，不过我发现这在这里同样没有帮助）

3. 使用重新加权的 KL 项（重要性采样）
   - 原式：`kl_loss = kl_coeff * torch.mean(logps - ref_logps)`
   - 新式：`kl_loss = kl_coeff * torch.mean(torch.exp(new_logps - old_logps) * (new_logps - ref_logps))`

4. 加入一个较小的格式奖励，防止优势值塌缩到零；下一节将讨论格式奖励

- 不过，如前所述，在数学数据上训练时，通常建议不使用 KL 项（Dr. GRPO、Olmo 3 和 DeepSeek V3.2 均如此建议）
  - 这也能降低代码复杂度和资源需求


&nbsp;
## 7.6 加入显式格式奖励

- 到目前为止只使用了单一的正确性奖励；本节将像 DeepSeek-R1 一样加入第二个格式奖励。

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F17_raschka.webp?1" width=400>

&nbsp;
### 7.6.1 使用 `<think>` 词元

- 一些推理模型（如 DeepSeek-R1 和 Qwen3）会使用所谓的 `<think>` 和 `</think>` 词元
- 这并非必需，但有助于鼓励模型将中间推理与最终答案分开，并强制采用一致的输出结构
- 首先加载之前使用的基础模型分词器：


In [ ]:
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import load_model_and_tokenizer

device = get_device()
model, tokenizer_base = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

- 可以看到，它将 `<think>` 拆分为多个子词词元，这说明 `<think>` 并不在词表中：

In [ ]:
print(tokenizer_base.encode("<think>"))

In [ ]:
for i in [13708, 766, 29]:
    print(tokenizer_base.decode([i]))

- 分词器中有一些空的占位槽，因此可以按如下方式将 `<think>` 词元加入词表：

In [ ]:
tokenizer_base._tok.add_special_tokens(
    ["<tool_response>", "</tool_response>", "<think>", "</think>"]
)

In [ ]:
print(tokenizer_base.encode("<think>"))

In [ ]:
print(tokenizer_base.encode("</think>"))

- 上面还加入了 `"<tool_response>"` 和 `"</tool_response>"` 词元，以便与稍后讨论的 Qwen3 推理变体分词器保持一致。

- 模型也支持这些词元，因为它的词表大小为 151936，可处理从 0 到 151935 的词元 ID。

In [ ]:
print("Vocabulary size:", model.tok_emb.weight.shape[0])

- 或者，推理模型变体的分词器已经原生支持这些 `<think>` 词元。

- 与第 2 章类似，可以单独加载该分词器：

In [ ]:
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer
from reasoning_from_scratch.qwen3 import download_qwen3_small

download_qwen3_small(
    kind="reasoning", tokenizer_only=True, out_dir="qwen3"
)

tokenizer_path = Path("qwen3") / "tokenizer-reasoning.json"
tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

In [ ]:
print(tokenizer.encode("<think>"))

In [ ]:
print(tokenizer.encode("</think>"))

- 接下来可以编写一个奖励函数，检查 LLM 是否使用了 `<think>` 和 `</think>` 标签
- 如果输出中同时出现 `<think>` 和 `</think>` 且顺序正确，该函数返回 1.0；否则返回 0.0


<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F18_raschka.webp?1" width=600>

In [ ]:
def reward_format(
    token_ids,
    prompt_len,
    start_think_id=151667,
    end_think_id=151668,
):
    try:
        gen = token_ids[prompt_len:].tolist()
        return float(
            gen.index(start_think_id) < gen.index(end_think_id)
        )
    except ValueError:
        return 0.0

In [ ]:
prompt = "Calculate ..."
rollout = "Let's ... <think> ... </think> ..."
token_ids = tokenizer.encode(prompt + rollout)

reward_format(
    token_ids=torch.tensor(token_ids),
    prompt_len=len(tokenizer.encode(prompt))
)

- 随后可将格式奖励作为额外奖励信号来训练模型
- 例如，可以按如下方式把 `compute_grpo_loss` 修改为 `compute_grpo_loss_plus_format_reward`：


In [ ]:
def compute_grpo_loss_plus_format_reward(
    # ...
    format_reward_weight=1.0,  # 新增
):
    # ...

        logp = sequence_logprob(model, token_ids, prompt_len)  # noqa: F821, F841
        rlvr_reward = reward_rlvr(text, example["answer"])  # noqa: F821
        format_reward = reward_format(token_ids, prompt_len)         # 新增  # noqa: F821
        reward = rlvr_reward + format_reward_weight * format_reward  # 新增  # noqa: F841

    # ...

- 此外，还需要修改 `render_prompt`，指示模型输出这些 `<think>` 词元：

In [ ]:
def render_prompt_with_think_tokens(prompt):
    template = (
        "You are a helpful math assistant.\n"
        "When solving the problem, first write your reasoning inside <think> and </think> tags.\n"
        "Then write the final result on a new line in the exact format:\n"
        "\\boxed{ANSWER}\n\n"
        f"Question:\n{prompt}\n\nAnswer:"
    )
    return template

&nbsp;
### 7.6.2 训练模型生成 `<think>` 词元

- 可以按如下方式从附加材料下载修改后的脚本：

In [ ]:
download_from_github(
    "ch07/03_rlvr_grpo_scripts_advanced/7_6_plus_format_reward.py"
)

- 如果直接用 GRPO 训练基础模型输出 `<think>` 词元，结果会很差，因为基础模型此前从未见过这些词元
- 更合理的做法是先使用包含 `<think>` 词元的数据对模型进行预训练或指令微调（下一章将介绍指令微调）
- 因此，本节训练已有的“推理”模型变体，它已经熟悉 `<think>` 词元


- 与之前类似，可以按如下方式运行（请注意，该脚本已配置为使用“推理”模型变体，而不是“基础”模型）：

```bash
uv run 7_6_plus_format_reward.py \
    --steps 500  \
    --max_new_tokens 1024
```


- 同样，由于运行实验可能耗费较多资源和成本，可以下载以下日志文件进行分析：

In [ ]:
download_from_github(
    "ch07/02_logs/7_6_plus_format_reward_metrics.csv"
)

In [ ]:
plot_grpo_metrics(
    "7_6_plus_format_reward_metrics.csv",
    columns=["loss", "reward_avg", "avg_response_len", "eval_acc"],
    #save_as="18.pdf"
)

> **关于 `loss` 和 `kl_loss` 的说明**
>
> - 本实验使用 `7_6_plus_format_reward.py`，默认设置为 `--kl_coeff 0.0` 和 `--inner_epochs 1`
> - 因为 `--kl_coeff` 设为 `0.0`，KL 惩罚被禁用，所以记录的 `kl_loss` 在整个运行期间都保持为 `0`
>
> - 如果要使用第 7.5 节中的相同设置启用 KL 正则化，请运行：
>
> ```bash
> uv run 7_6_plus_format_reward.py \
>     --steps 500 \
>     --max_new_tokens 1024 \
>     --kl_coeff 0.001 \
>     --inner_epochs 2
> ```
>
> - 使用第 7.6 节的默认设置时，记录的 `loss` 也为 `0`
> - 这是因为该运行只执行一次内部更新（`--inner_epochs 1`），并将模型与同一步骤中自身的副本进行比较
> - 因此策略比率从 `1` 开始，而优势值经过归一化后均值为 `0`，所以标量策略损失的计算结果为 `0`
>
> - 因此，这里的 `loss = 0` 是默认设置导致的计算现象
> - 换言之，它主要反映该运行只使用单个内部训练轮次且没有 KL 项，使记录的标量损失缺乏信息量
> - 此时更值得监控的是 `reward_avg`、`format_reward_avg`、`adv_std` 和 `entropy_avg` 等指标


In [ ]:
plot_grpo_metrics(
    "7_6_plus_format_reward_metrics.csv",
    columns=["reward_avg", "format_reward_avg", "adv_std", "entropy_avg"],
    #save_as="19.pdf"
)

- 可以看到，模型最初有所改进，但在前 100 步之后开始变差；这可能与模型在这 100 步后输出的回复变短有关
- 一种可能的解释是，如上所示，模型仅遵循 `<think>` 标签格式就已获得过多奖励，因而不再充分关注答案正确性
- 可以把默认的 `format_reward_weight` 从 1.0 降至 0.1，以减小格式奖励的权重
- 另一种改进方式是让格式奖励具有条件性，即只有答案正确时才给予格式奖励（目前无论答案正确与否都会独立给予）


```python
if conditional_reward:
    format_reward *= rlvr_reward
reward = rlvr_reward + format_reward_weight * format_reward
```

&nbsp;
## 7.7 更多 GRPO 改进、技巧和诀窍

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch07/CH07_F21_raschka.webp?3" width=400>

- DeepSeek-R1 成功后的几个月里，许多研究者提出了对原始 GRPO 算法的改进，以同时提高稳定性和性能
- 本章篇幅已经很长，因此这里只简要列出每项变化或改进
- 更多信息和代码请参阅[附加材料](../03_rlvr_grpo_scripts_advanced/)

1. 过滤零梯度信号（[Yu 等人，2025，DAPO](https://arxiv.org/abs/2503.14476)）
2. 主动采样（DAPO）
3. 词元级损失（DAPO）
4. 不使用 KL 损失（DAPO 和 [Liu 等人，2025，Dr. GRPO](https://arxiv.org/abs/2503.20783)）
5. 提高裁剪上界（DAPO）
6. 截断重要性采样（[Yao 等人，2025](https://fengyao.notion.site/off-policy-rl)）
7. 不进行标准差归一化（Dr. GRPO）
8. 根据领域调整 KL 强度；数学领域设为零（[DeepSeek V3.2](https://arxiv.org/abs/2512.02556)）
9. 重新加权的 KL（DeepSeek V3.2）
10. 离策略序列掩码（DeepSeek V3.2）
11. 为 top-p / top-k 保留采样掩码（DeepSeek V3.2）
12. 保留原始 GRPO 的优势值归一化（DeepSeek V3.2）
13. 聚合之前，对每种奖励分别进行组内归一化（[Liu 等人，2026，GDPO](https://arxiv.org/abs/2601.05242)）
14. 序列级重要性采样和裁剪（[Zheng 等人，2025，GSPO](https://arxiv.org/abs/2507.18071)）
15. 裁剪重要性采样权重，而不是词元更新（[MiniMax 等人，2025，CISPO](https://arxiv.org/abs/2506.13585)）


&nbsp;
## 7.8 小结

- 本节没有代码。